# 04 - Statistical Analysis

## Customer Intelligence Platform

This notebook applies rigorous hypothesis testing to validate EDA observations.
As a Statistics graduate, this is where you demonstrate formal analytical rigor.

### Methods Used
- **Categorical Features**: Chi-Square Test + Cramer's V (effect size)
- **Numerical Features**: Welch's t-Test + Cohen's d (effect size)
- **Multiple Testing**: Bonferroni and Benjamini-Hochberg corrections

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.load_data import load_clean
from src.config import TARGET, CATEGORICAL_FEATURES, NUMERICAL_FEATURES
from src.statistics import (
    run_all_categorical_tests,
    run_all_numerical_tests,
    summarize_all_tests,
)


In [ ]:
df = load_clean()


## 1. Categorical Variable Tests

For each categorical feature, we test whether its distribution differs
significantly between churned and retained customers using the **Chi-Square
test of independence**.

**Effect Size**: Cramer's V
- < 0.1: Negligible
- 0.1-0.3: Small
- 0.3-0.5: Medium
- >= 0.5: Large


In [ ]:
cat_results = run_all_categorical_tests(df, CATEGORICAL_FEATURES, TARGET)
cat_results[["feature", "chi2", "p_value", "cramers_v", "significant", "effect_interpretation"]]


### Interpretation

The features are ranked by Cramer's V (practical significance), not just p-value (statistical significance).

**Key Insight**: Many features are statistically significant (p < 0.05) but have small practical effect sizes.
This distinction is critical for modeling - a feature can be statistically significant with thousands of
observations but practically irrelevant.

---

## 2. Numerical Variable Tests

For numerical features, we use **Welch's t-test** (which handles unequal variances)
to compare means between churned and retained groups.

**Effect Size**: Cohen's d
- < 0.2: Negligible
- 0.2-0.5: Small
- 0.5-0.8: Medium
- >= 0.8: Large


In [ ]:
num_results = run_all_numerical_tests(df, NUMERICAL_FEATURES, TARGET)
num_results[["feature", "t_stat", "p_value", "cohens_d", "mean_retained", "mean_churned", "effect_interpretation"]]


---

## 3. Multiple Testing Correction

When conducting many hypothesis tests simultaneously, the probability of at least
one false positive increases. We apply two corrections:

1. **Bonferroni**: Most conservative. Divides alpha by number of tests.
2. **Benjamini-Hochberg**: Controls the False Discovery Rate (FDR). Less conservative.

Most student projects ignore this. Including it demonstrates advanced statistical literacy.


In [ ]:
# Show corrected p-values for categorical tests
correction_df = cat_results[["feature", "p_value", "p_bonferroni", "p_bh", "cramers_v"]].copy()
correction_df["still_significant_bonf"] = correction_df["p_bonferroni"] < 0.05
correction_df["still_significant_bh"] = correction_df["p_bh"] < 0.05
correction_df


In [ ]:
# Show corrected p-values for numerical tests
num_correction = num_results[["feature", "p_value", "p_bonferroni", "p_bh", "cohens_d"]].copy()
num_correction["still_significant_bonf"] = num_correction["p_bonferroni"] < 0.05
num_correction["still_significant_bh"] = num_correction["p_bh"] < 0.05
num_correction


### Multiple Testing Findings

After applying corrections, we assess which features retain significance:

- **Bonferroni** (most conservative): Features with very strong effects survive.
- **Benjamini-Hochberg**: More lenient, controls false discovery rate at 5%.

This analysis strengthens our confidence in the top predictors identified during EDA.

---

## 4. Feature Importance Ranking (Statistical)

Combining effect sizes from both categorical and numerical tests:


In [ ]:
# Combined ranking
cat_rank = cat_results[["feature", "cramers_v", "effect_interpretation"]].rename(
    columns={"cramers_v": "effect_size"}
)
cat_rank["metric"] = "Cramer's V"

num_rank = num_results[["feature", "cohens_d", "effect_interpretation"]].rename(
    columns={"cohens_d": "effect_size"}
)
num_rank["effect_size"] = num_rank["effect_size"].abs()
num_rank["metric"] = "Cohen's d"

combined = pd.concat([cat_rank, num_rank]).sort_values("effect_size", ascending=False).reset_index(drop=True)
combined


## Summary

### Statistically Confirmed Churn Drivers

| Rank | Feature | Effect Size | Metric | Practical Significance |
|------|---------|------------|--------|----------------------|
| 1 | Contract | Large | Cramer's V | Month-to-month strongly predicts churn |
| 2 | Tenure Months | Large | Cohen's d | Shorter tenure = higher churn risk |
| 3 | Online Security | Medium | Cramer's V | Security subscription reduces churn |
| 4 | Tech Support | Medium | Cramer's V | Support subscription reduces churn |
| 5 | Internet Service | Medium | Cramer's V | Fiber customers churn more |
| 6 | Monthly Charges | Medium | Cohen's d | Higher charges increase churn |
| 7 | Dependents | Small-Medium | Cramer's V | Family customers are stable |

### Statistical Rigor Applied
- All tests account for unequal variances (Welch's t-test)
- Effect sizes reported alongside p-values
- Multiple testing corrections applied (Bonferroni + BH)
- Practical significance distinguished from statistical significance
